# Phase 2b — Spatial instance-count readout
Phase 2's pooled probe was blind to spatial count. Here we read the count *spatially*: turn each up-block **self-attention** feature map into a saliency map, threshold + count blobs = an **internal instance count**, and see where/when it diverges from the requested number.

**Question:** is the count wrong from the earliest step (set by the noise layout) or does it drift later (dynamic failure)?

**Runtime:** GPU.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
from src.prompts import generate_grid
from src.pipeline import (load_sdxl, catalog_attention_sites,
                          select_probe_sites, generate_and_capture)
from src.spatial import featuremap_saliency, count_peaks, find_peaks
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase2b.yaml')
raw = yaml.safe_load(open('configs/phase2b.yaml'))
PK = dict(sigma=raw['peak_sigma'], min_distance=raw['peak_min_distance'],
          thresh_rel=raw['peak_thresh_rel'])
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
print(len(grid), 'images; steps', cfg.capture_steps, '; peak params', PK)

In [ ]:
pipe = load_sdxl()
det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if 'up_blocks' in s and s.endswith('attn1')]
print(len(sites), 'up-block self-attention (image-side) sites:')
print(sites)

In [ ]:
# Capture SALIENCY MAPS (not pooled) at each up-block attn1 site x step.
sal_store = {st: {s: [] for s in sites} for st in cfg.capture_steps}
images, rows = [], []
for i, p in enumerate(grid):
    img, snaps = generate_and_capture(pipe, p.text, p.seed, sites,
                                      cfg.capture_steps, cfg.num_inference_steps,
                                      reducer=featuremap_saliency)
    rendered = count_from_detections(det.detect(img, [p.obj]), p.obj,
                                     cfg.score_threshold)
    images.append(img)
    rows.append({'obj': p.obj, 'count': p.count, 'seed': p.seed, 'rendered': rendered})
    for st in cfg.capture_steps:
        for s in sites:
            sal_store[st][s].append(snaps.get(st, {}).get(s))
    if (i + 1) % 20 == 0:
        print(f'{i+1}/{len(grid)}')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase2b_counts.csv', index=False)
df.head()

In [ ]:
# Internal instance count = mean PEAK-count over up-block sites, per step.
req = df['count'].to_numpy(float); ren = df['rendered'].to_numpy(float)
def inst_at(st, s):
    return np.array([count_peaks(m, **PK) if m is not None
                     else np.nan for m in sal_store[st][s]])
inst = {st: np.nanmean(np.vstack([inst_at(st, s) for s in sites]), axis=0)
        for st in cfg.capture_steps}
def safecorr(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = ~(np.isnan(a) | np.isnan(b))
    if m.sum() < 3 or np.std(a[m]) == 0 or np.std(b[m]) == 0: return np.nan
    return float(np.corrcoef(a[m], b[m])[0, 1])
print('step :  corr(internal, requested) | corr(internal, rendered) | mean internal')
for st in cfg.capture_steps:
    print(f'{st:>4} : {safecorr(inst[st], req):>25.2f} | '
          f'{safecorr(inst[st], ren):>18.2f} | {np.nanmean(inst[st]):.2f}')

In [ ]:
# Money plot: mean internal instance count vs requested count, per step.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5))
for st in cfg.capture_steps:
    means = [np.nanmean(inst[st][req == c]) for c in cfg.counts]
    ax.plot(cfg.counts, means, marker='o', label=f'step {st}')
ax.plot(cfg.counts, cfg.counts, 'k--', label='y=x (internal = requested)')
ax.set_xlabel('requested count'); ax.set_ylabel('mean internal instance count')
ax.set_title('Does the internal spatial count track the request?')
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig('results/phase2b_internal_vs_requested.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Eyeball: validate that the saliency segments objects. image | saliency.
site0, st0 = sites[0], (15 if 15 in cfg.capture_steps else cfg.capture_steps[len(cfg.capture_steps)//2])
order = np.argsort(df['count'].to_numpy())
pick = order[np.linspace(0, len(order) - 1, 6).astype(int)]
fig, axes = plt.subplots(len(pick), 2, figsize=(7, 3 * len(pick)))
for row, idx in enumerate(pick):
    sal = sal_store[st0][site0][idx]
    pk = find_peaks(sal, **PK) if sal is not None else np.empty((0, 2))
    axes[row, 0].imshow(images[idx]); axes[row, 0].axis('off')
    axes[row, 0].set_title(f"asked {df['count'][idx]} rendered {df['rendered'][idx]}", fontsize=9)
    axes[row, 1].imshow(sal, cmap='magma'); axes[row, 1].axis('off')
    if len(pk):
        axes[row, 1].scatter(pk[:, 1], pk[:, 0], c='cyan', s=60, marker='x')
    axes[row, 1].set_title(f'saliency+peaks @ {site0.split(".")[1]} step {st0} | peaks={len(pk)}', fontsize=8)
plt.tight_layout()
plt.savefig('results/phase2b_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **corr(internal, requested) high at early steps, then falling** = the layout starts honoring the number and loses it as denoising proceeds -> the count is lost DURING allocation; the step where it drops is where.
- **corr(internal, requested) low from the earliest step** = the spatial count is wrong from the start -> set by the initial noise layout, not drift.
- **corr(internal, rendered) rising toward late steps** = the internal blob-count converges to the eventual (wrong) output -> our readout is real and the error crystallizes by that step.
- **Money plot:** lines on the diagonal at some step = internal count matches the request there; lines flattening ABOVE the diagonal = over-allocation (too many blobs), matching Phase 1's over-generation.
- **Eyeball first!** If the saliency maps don't land on the objects, raise/lower `saliency_thresh` in `configs/phase2b.yaml` before trusting the counts. The best-tracking site+step becomes the Phase 2c causal-patch target.